{Nº41) Составить на ПОСП (GPSS, python) алгоритм агрегативной модели процесса функционирования системы:
Линия связи состоит из 2х каналов (основного и резервного) и общего накопителя. Сообщения поступают от 2х источников. Приоритет сообщений первого источника выше, чем второго. От первого источника интервал поступления между сообщениями в диапазоне [8..10], от второго - [1...3] мс. При нормальной работе сообщения передаются по основному каналу за 4.5 мс. В основном канале происходят сбои через 2..3 сек.
Если сбой происходит во время передачи, то сообщение передаётся по резервному каналу с самого начала.
Восстановление основного канала занимает [16...30] мс. Во время отсутствия основного канала сообщения передаются по резервному каналу. Оценить загрузку каналов, количество переданных по ним сообщений, характеристики очереди сообщений. Закон распределения всех стохастических параметров равномерный.

In [41]:
import simpy
import random

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [42]:
class PriorityQueue:
    def __init__(self, env):
        self.env = env
        self.items = []

    def put(self, msg):
        self.items.append(msg)

    def get(self):
        # Сначала ищем сообщения от источника 1 (приоритет)
        for i, (src, t) in enumerate(self.items):
            if src == 1:
                return self.items.pop(i)
        # Если нет — берем первое попавшееся
        return self.items.pop(0) if self.items else None

    def __len__(self):
        return len(self.items)

In [43]:
def message_generator(env, queue, source_id, interval_range):
    while True:
        yield env.timeout(random.uniform(*interval_range))  # ждем до следующего сообщения
        queue.put((source_id, env.now))  # кладем сообщение в очередь

In [44]:
def reserve_channel(env, msg, stats, queue_stats):
    start = env.now
    yield env.timeout(4.5)  # передача по резерву
    stats['reserve'] += 1
    queue_stats['wait_times'].append(start - msg[1])

In [45]:
def main_channel(env, queue, stats, queue_stats):
    while True:
        # Время до сбоя
        work_time = random.uniform(2000, 3000)
        fail_time = env.now + work_time
        while env.now < fail_time:
            if len(queue) == 0:
                yield env.timeout(0.1)
                continue
            msg = queue.get()
            start = env.now
            # Проверяем, не случится ли сбой во время передачи
            if env.now + 4.5 > fail_time:
                # Сбой во время передачи — передаем по резерву
                yield env.timeout(fail_time - env.now)  # до сбоя
                env.process(reserve_channel(env, msg, stats, queue_stats))
                break  # выходим на восстановление
            else:
                yield env.timeout(4.5)  # передача по основному
                stats['main'] += 1
                queue_stats['wait_times'].append(start - msg[1])
        # Восстановление основного канала
        repair_time = random.uniform(16, 30)
        yield env.timeout(repair_time)

In [46]:
stats = {'main': 0, 'reserve': 0}
queue_stats = {'wait_times': [], 'max_len': 0}

In [47]:
def queue_monitor(env, queue, queue_stats):
    while True:
        queue_stats['max_len'] = max(queue_stats['max_len'], len(queue))
        yield env.timeout(0.1)

In [48]:
env = simpy.Environment()
queue = PriorityQueue(env)

# Запуск генераторов сообщений
env.process(message_generator(env, queue, 1, (8, 10)))
env.process(message_generator(env, queue, 2, (1, 3)))

# Запуск мониторинга очереди
env.process(queue_monitor(env, queue, queue_stats))

# Запуск основного канала
env.process(main_channel(env, queue, stats, queue_stats))

# Запуск симуляции на 40000 мс (пример)
env.run(until=40000)

In [51]:
# Оценка загрузки каналов и характеристик очереди

# Вычисление загрузки основного канала
main_channel_load = stats['main'] / (stats['main'] + stats['reserve']) if (stats['main'] + stats['reserve']) > 0 else 0

# Вычисление загрузки резервного канала
reserve_channel_load = stats['reserve'] / (stats['main'] + stats['reserve']) if (stats['main'] + stats['reserve']) > 0 else 0

# Вывод результатов
print("Загрузка основного канала:", main_channel_load)
print("Загрузка резервного канала:", reserve_channel_load)
print("Количество переданных сообщений по основному каналу:", stats['main'])
print("Количество переданных сообщений по резервному каналу:", stats['reserve'])
print("Максимальная длина очереди:", queue_stats['max_len'])
if queue_stats['wait_times']:
    print("Среднее время ожидания в очереди:", sum(queue_stats['wait_times']) / len(queue_stats['wait_times']))
else:
    print("Нет сообщений в очереди")


Загрузка основного канала: 0.9982991268851343
Загрузка резервного канала: 0.001700873114865631
Количество переданных сообщений по основному каналу: 8804
Количество переданных сообщений по резервному каналу: 15
Максимальная длина очереди: 15644
Среднее время ожидания в очереди: 7732.604529945598
